In [25]:
!pip install langchain langgraph langchain-community pypdf sqlite-utils langchain-groq pytesseract Pillow pdf2image

In [56]:
#from google.colab import drive
#drive.mount('/Data')

In [27]:
import os
import sqlite3
from datetime import datetime
import json
import re # Added for parsing LLM JSON output

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

from langchain_community.document_loaders import PyPDFLoader

from langgraph.graph import StateGraph, END
from typing import TypedDict

**OCR Dependencies**

In [28]:
!sudo apt-get update
!sudo apt-get install -y tesseract-ocr
!sudo apt-get install -y poppler-utils

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 

Setup Groq API **Key**

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
# 2. Setup API Keys
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# Optional for LangSmith Tracking
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "RAG Basics"

In [31]:
# Define a confidence threshold
CONFIDENCE_THRESHOLD = 0.75 # You can adjust this value based on your needs

Initialize Groq **LLM**

In [32]:

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0
)

**Audit Logging Agent**

In [33]:
def audit_log(action, details):

    log_entry = f"{datetime.now()} | {action} | {details}\n"

    with open("audit_log.txt", "a") as f:
        f.write(log_entry)

    print("Audit:", log_entry)

**Database Creation**

In [34]:
conn = sqlite3.connect("cease_requests.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS cease_requests (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_name TEXT,
    received_date TEXT,
    extracted_details TEXT
)
""")

conn.commit()

**Define Agent Shared State**

In [35]:
class AgentState(TypedDict):

    document_path: str
    document_text: str
    classification: str
    extracted_details: str
    confidence: float # New: Added confidence score

**Document Loader Agent**

In [36]:
from pdf2image import convert_from_path
import pytesseract
from PIL import Image

def document_loader_agent(state):
    path = state["document_path"]
    text = ""

    # Try loading text with PyPDFLoader first
    try:
        loader = PyPDFLoader(path)
        docs = loader.load()
        py_pdf_text = "\n".join([d.page_content for d in docs])
        print(f"DEBUG: PyPDFLoader extracted (length: {len(py_pdf_text)}): '{py_pdf_text[:100].replace('\n', ' ')}...' (truncated)")

        if py_pdf_text.strip() == "": # If PyPDFLoader extracts no text, it might be a scanned document
            print(f"DEBUG: PyPDFLoader extracted no text from {path}. Raising error to attempt OCR...")
            raise ValueError("No text extracted by PyPDFLoader, falling back to OCR.")
        else:
            text = py_pdf_text
            print(f"DEBUG: PyPDFLoader successfully extracted text. Skipping OCR for {path}.")
    except Exception as e:
        # Fallback to OCR if PyPDFLoader fails or extracts no text
        print(f"DEBUG: Fallback to OCR for {path} due to: {e}")
        try:
            images = convert_from_path(path)
            ocr_text_pages = []
            for i, image in enumerate(images):
                print(f"DEBUG: OCR processing page {i+1} of {len(images)} for {path}...")
                page_text = pytesseract.image_to_string(image)
                ocr_text_pages.append(page_text)
            text = "\n".join(ocr_text_pages)
            print(f"DEBUG: OCR extracted (length: {len(text)}): '{text[:100].replace('\n', ' ')}...' (truncated)")

            if text.strip() == "":
                 print(f"WARNING: OCR also extracted no text from {path}.")

        except Exception as ocr_e:
            print(f"ERROR: OCR failed for {path}: {ocr_e}")
            text = "" # Ensure text is empty if OCR also fails

    if text.strip() == "":
        audit_log("DOCUMENT_LOAD_FAILED_EMPTY", path)
        print(f"WARNING: No content extracted from {path} even after OCR attempt.")
    else:
        audit_log("DOCUMENT_LOADED", path)
        print(f"DEBUG: Successfully extracted text from {path} (length: {len(text)}). Final text preview: '{text[:100].replace('\n', ' ')}...' (truncated)")

    return {
        "document_text": text
    }

**Classification Agent Prompt**

In [37]:
classification_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are a legal assistant.

Classify the document into one category:
Cease – request is a formal demand asking someone to stop (cease) doing a specific action and not start it again (desist) in the future
Irrelevant – unrelated document
Uncertain – unclear or incomplete

Provide your response in JSON format with two keys:
1. "classification": The classified category (Cease, Irrelevant, Uncertain)
2. "confidence": A score from 0.0 to 1.0 indicating your certainty about the classification.

Document:
{text}

JSON Output:
"""
)

**Classification Agent**

In [38]:
def classification_agent(state):

    text = state["document_text"]

    prompt = classification_prompt.format(text=text[:3000])

    response = llm.invoke(prompt)

    raw_response_content = response.content

    classification = "Uncertain"
    confidence = 0.0

    try:
        json_to_parse = raw_response_content

        # First, try to extract JSON wrapped in ```json ... ```
        json_match = re.search(r"```json\s*(.*?)\s*```", raw_response_content, re.DOTALL)
        if json_match:
            json_to_parse = json_match.group(1)
        else:
            # If not in code block, try to find the JSON object directly
            start_idx = raw_response_content.find('{')
            end_idx = raw_response_content.rfind('}')
            if start_idx != -1 and end_idx != -1 and start_idx < end_idx:
                json_to_parse = raw_response_content[start_idx : end_idx + 1]
            # else, json_to_parse remains raw_response_content, and json.loads will likely fail

        print(f"DEBUG: json_to_parse before loads: {repr(json_to_parse)}") # New debug print
        parsed_json = json.loads(json_to_parse)
        classification = parsed_json.get("classification", "Uncertain").strip()
        confidence = float(parsed_json.get("confidence", 0.0)) # Ensure confidence is a float
    except (json.JSONDecodeError, ValueError) as e:
        print(f"DEBUG: Failed to parse JSON from LLM response: {e}")
        print(f"DEBUG: Raw LLM response: {raw_response_content}")
        classification = "Uncertain" # Default if parsing fails
        confidence = 0.0 # Default confidence if parsing fails

    print(f"DEBUG: Raw LLM response: '{raw_response_content}'")
    print(f"DEBUG: Extracted classification: '{classification}', Confidence: {confidence}")

    audit_log("CLASSIFICATION", f"{classification} (Confidence: {confidence})")

    return {
        "classification": classification,
        "confidence": confidence
    }

**Extract Agent Prompt**

In [39]:
extract_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Extract key details from the cease request.

Return:

Customer Name
Address
Reason for request
Requested stop communication type

Document:
{text}
"""
)

**Extract Agent**

In [40]:
def extract_agent(state):

    text = state["document_text"]

    prompt = extract_prompt.format(text=text[:3000])

    response = llm.invoke(prompt)

    details = response.content

    audit_log("DETAILS_EXTRACTED", details)

    return {
        "extracted_details": details
    }

**Database Agent**

In [41]:
def database_agent(state):

    cursor.execute(
        """
        INSERT INTO cease_requests
        (document_name, received_date, extracted_details)
        VALUES (?, ?, ?)
        """,
        (
            state["document_path"],
            str(datetime.now()),
            state["extracted_details"]
        )
    )

    conn.commit()

    audit_log("DATABASE_WRITE", state["document_path"])

    return {}

**Archive Agent**

In [42]:
def archive_agent(state):

    entry = f"{datetime.now()} | {state['document_path']}\n"

    with open("archive.txt", "a") as f:
        f.write(entry)

    audit_log("ARCHIVED", state["document_path"])

    return {}

**Human In The Loop Agent**

In [43]:
def hitl_agent(state):

    print("\n⚠️ Document requires human review\n")

    print(state["document_text"][:1000])

    decision = input("Enter decision (Cease / Irrelevant): ")

    audit_log("HUMAN_DECISION", decision)

    return {
        "classification": decision
    }

**Route Decision Agent**

In [44]:
def route_decision(state):
    classification = state["classification"]
    confidence = state.get("confidence", 0.0) # Get confidence, default to 0 if not present
    print(f"DEBUG: route_decision received classification: {repr(classification)}, Confidence: {confidence}")

    if classification == "Cease":
        if confidence >= CONFIDENCE_THRESHOLD:
            return "extract"
        else:
            print(f"DEBUG: route_decision returning 'hitl' for 'Cease' due to low confidence ({confidence} < {CONFIDENCE_THRESHOLD})")
            return "hitl"

    elif classification == "Irrelevant":
        if confidence >= CONFIDENCE_THRESHOLD:
            return "archive"
        else:
            print(f"DEBUG: route_decision returning 'hitl' for 'Irrelevant' due to low confidence ({confidence} < {CONFIDENCE_THRESHOLD})")
            return "hitl"

    else: # Classification is 'Uncertain' or anything else
        print(f"DEBUG: route_decision returning 'hitl' because classification was '{classification}' (Confidence: {confidence})")
        return "hitl"

**StateGraph Nodes & Edges Declation and Complile**

In [45]:
def route_after_hitl(state):
    # Route based on the human's decision
    classification = state["classification"]
    print(f"DEBUG: route_after_hitl received classification from human: {repr(classification)}")
    if classification == "Cease":
        return "extract"
    elif classification == "Irrelevant":
        return "archive"
    else:
        # If human input is invalid, loop back to hitl for correction or raise error
        print("Warning: Human entered invalid classification after HITL. Looping back to HITL.")
        print(f"DEBUG: route_after_hitl returning 'hitl' because human input was not 'Cease' or 'Irrelevant'. Input: {repr(classification)}")
        return "hitl"

builder = StateGraph(AgentState)

builder.add_node("loader", document_loader_agent)
builder.add_node("classifier", classification_agent)
builder.add_node("extract", extract_agent)
builder.add_node("database", database_agent)
builder.add_node("archive", archive_agent)
builder.add_node("hitl", hitl_agent)

builder.set_entry_point("loader")

builder.add_edge("loader", "classifier")

builder.add_conditional_edges(
    "classifier",
    route_decision,
    {
        "extract": "extract",
        "archive": "archive",
        "hitl": "hitl"
    }
)

builder.add_edge("extract", "database")

builder.add_edge("database", END)
builder.add_edge("archive", END)

builder.add_conditional_edges(
    "hitl",
    route_after_hitl,
    {
        "extract": "extract",
        "archive": "archive",
        "hitl": "hitl" # Loop back to hitl if human input is unexpected
    }
)

graph = builder.compile()

**Batch Processing & Graph Invoking**


## 🔁 Enhanced Features: Memory + HITL Feedback Loop

### ✅ Memory
- Stores extracted entities and decisions from previous documents
- Used to improve future document analysis

### ✅ HITL (Human-in-the-Loop)
- Any human feedback/comments are stored
- Future documents are analyzed using this feedback

This creates a continuous learning pipeline.


In [46]:

# Simple in-memory store (can be replaced with DB / vector store)

class MemoryStore:
    def __init__(self):
        self.data = []
        self.hitl_feedback = []

    def add_document_data(self, doc_id, extracted_data):
        self.data.append({
            "doc_id": doc_id,
            "data": extracted_data
        })

    def add_hitl_feedback(self, doc_id, comments):
        self.hitl_feedback.append({
            "doc_id": doc_id,
            "comments": comments
        })

    def get_context(self):
        return {
            "past_data": self.data,
            "feedback": self.hitl_feedback
        }

memory_store = MemoryStore()


In [47]:

def process_with_memory(doc_id, extracted_data):
    # Store extracted data
    memory_store.add_document_data(doc_id, extracted_data)

    # Get past context
    context = memory_store.get_context()

    print("Using Memory Context:")
    print(context)

    return context


def hitl_review(doc_id):
    comments = input("Enter HITL comments for this document: ")
    memory_store.add_hitl_feedback(doc_id, comments)
    print("Feedback stored successfully!")


In [53]:
def batch_process_documents(directory_path):
    processed_count = 0
    for filename in os.listdir(directory_path):
        if filename.endswith(".pdf"):
            document_path = os.path.join(directory_path, filename)
            print(f"\n--- Processing document: {document_path} ---")
            input_data = {
                "document_path": document_path
            }
            try:
                graph.invoke(input_data)
                processed_count += 1
            except Exception as e:
                print(f"Error processing {document_path}: {e}")
                audit_log("BATCH_PROCESS_ERROR", f"Error processing {document_path}: {e}")

    print(f"\n--- Batch processing complete. {processed_count} documents processed. ---")




--- Processing document: /content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf ---
DEBUG: PyPDFLoader extracted (length: 3153): 'CEASE AND DESIST LETTER Priya Nair Professional Photographer Priya Nair Photography LLC 482 Oak Ridg...' (truncated)
DEBUG: PyPDFLoader successfully extracted text. Skipping OCR for /content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf.
Audit: 2026-03-20 11:11:41.788290 | DOCUMENT_LOADED | /content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf

DEBUG: Successfully extracted text from /content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf (length: 3153). Final text preview: 'CEASE AND DESIST LETTER Priya Nair Professional Photographer Priya Nair Photography LLC 482 Oak Ridg...' (truncated)
DEBUG: json_to_parse before loads: '{\n  "classification": "Cease",\n  "confidence": 0.98\n}'
DEBUG: Raw LLM response: '<think>
Okay, let's see. The user wants me to classify this document into one of three categ

In [60]:
# Example usage:
# In Google Drive created a Folder with the Name "Test"
batch_process_documents("/content/drive/MyDrive/Test/")


--- Processing document: /content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf ---
DEBUG: PyPDFLoader extracted (length: 3153): 'CEASE AND DESIST LETTER Priya Nair Professional Photographer Priya Nair Photography LLC 482 Oak Ridg...' (truncated)
DEBUG: PyPDFLoader successfully extracted text. Skipping OCR for /content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf.
Audit: 2026-03-20 11:24:59.798120 | DOCUMENT_LOADED | /content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf

DEBUG: Successfully extracted text from /content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf (length: 3153). Final text preview: 'CEASE AND DESIST LETTER Priya Nair Professional Photographer Priya Nair Photography LLC 482 Oak Ridg...' (truncated)
DEBUG: json_to_parse before loads: '{\n  "classification": "Cease",\n  "confidence": 1.0\n}'
DEBUG: Raw LLM response: '<think>
Okay, let's see. The user wants me to classify this document into one of three catego

**Writing Log to txt File**

In [54]:
with open("audit_log.txt", "r") as f:
    print(f.read())

2026-03-20 11:11:41.788290 | DOCUMENT_LOADED | /content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf
2026-03-20 11:11:42.554800 | CLASSIFICATION | Cease (Confidence: 0.98)
2026-03-20 11:11:43.800969 | DETAILS_EXTRACTED | <think>
Okay, let's see. The user wants me to extract key details from this cease and desist letter. The specific fields they need are Customer Name, Address, Reason for request, and Requested stop communication type.

First, I need to identify the customer. The letter is from Priya Nair to David Chen. But the customer here is the one making the request, so that's Priya Nair. Her address is listed as 482 Oak Ridge Drive, Austin, TX 78701. 

The reason for the request is copyright infringement. The letter mentions unauthorized use of photographs titled 'Urban Textures' on various platforms. The legal basis is under 17 U.S.C. sections 106, 501, and 504. 

For the requested stop communication type, the demands include ceasing all unlawful activity related 

**Results Printing from Database**

In [55]:
cursor.execute("SELECT * FROM cease_requests")

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, '/content/drive/MyDrive/Test/01_copyright_infringement_photography.pdf', '2026-03-20 11:11:43.803507', '<think>\nOkay, let\'s see. The user wants me to extract key details from this cease and desist letter. The specific fields they need are Customer Name, Address, Reason for request, and Requested stop communication type.\n\nFirst, I need to identify the customer. The letter is from Priya Nair to David Chen. But the customer here is the one making the request, so that\'s Priya Nair. Her address is listed as 482 Oak Ridge Drive, Austin, TX 78701. \n\nThe reason for the request is copyright infringement. The letter mentions unauthorized use of photographs titled \'Urban Textures\' on various platforms. The legal basis is under 17 U.S.C. sections 106, 501, and 504. \n\nFor the requested stop communication type, the demands include ceasing all unlawful activity related to the photographs. The specific communication types mentioned are the unauthorized use in websites, social media, and